# 02 · 链式法则与手推反向传播

> **本节属于 Part 1 · 数学与梯度直觉。**

上一节的梯度是我们直接写公式得到的。但当模型有很多层、很多参数时，怎么系统地求出"损失对每个参数的梯度"？答案是**反向传播 (backpropagation)**，它的数学核心就是**链式法则**。

本节我们**手推**两个例子的反向传播——先是一个神经元，再是一个两层网络——并用数值梯度检查和 PyTorch 双重验证。理解了这一节，你就理解了所有深度学习框架内部到底在算什么。

## 学习目标

- 彻底理解**链式法则**与"**上游梯度 × 局部梯度**"这一反向传播的核心口诀
- 手推并实现**单个神经元**的反向传播
- 手推并实现**两层神经网络**（向量化）的反向传播
- 用 `gradcheck` 与 `torch.autograd` 双重验证每一个梯度

## 直觉与数学原理

**复合函数求导 = 链式法则。** 如果 $L$ 依赖 $a$，$a$ 又依赖 $z$，$z$ 又依赖 $w$，那么

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial a}\cdot\frac{\partial a}{\partial z}\cdot\frac{\partial z}{\partial w}$$

把计算看成一张**计算图**：前向时从左到右算出每个中间值；反向时从右到左，把"**到目前为止的梯度（上游梯度）**"乘上"**这一步的局部导数**"，一路传回去。这就是反向传播。

**例子（一个神经元）**：

$$z = wx + b, \qquad a = \sigma(z), \qquad L = (a - y)^2$$

其中 $\sigma$ 是 sigmoid，其导数有个漂亮的形式 $\sigma'(z) = \sigma(z)\,(1-\sigma(z)) = a(1-a)$。按链式法则反向推：

$$\frac{\partial L}{\partial a} = 2(a-y),\quad \frac{\partial L}{\partial z} = \frac{\partial L}{\partial a}\,a(1-a),\quad \frac{\partial L}{\partial w} = \frac{\partial L}{\partial z}\,x,\quad \frac{\partial L}{\partial b} = \frac{\partial L}{\partial z}$$

## 从零手写实现（一）：单个神经元

我们对一组具体数值，**一步步**地做前向与反向。

In [ ]:
import numpy as np
from minitorch import set_seed, gradcheck

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

x, y = 1.5, 1.0          # 单个样本
w, b = 0.8, -0.2         # 参数

# ---------- 前向：从左到右算出每个中间量 ----------
z = w * x + b
a = sigmoid(z)
L = (a - y) ** 2
print(f"前向:  z={z:.4f},  a={a:.4f},  L={L:.4f}")

# ---------- 反向：从右到左，上游梯度 × 局部梯度 ----------
dL_da = 2 * (a - y)          # L = (a-y)^2
da_dz = a * (1 - a)          # sigmoid'
dL_dz = dL_da * da_dz        # 链式：累乘
dL_dw = dL_dz * x            # z = wx+b  ->  dz/dw = x
dL_db = dL_dz * 1.0          # dz/db = 1
print(f"反向:  dL/dw={dL_dw:.4f},  dL/db={dL_db:.4f}")

**验证**：用数值梯度检查，确认我们手推的 `dL/dw, dL/db` 正确。

In [ ]:
def loss_neuron(p):
    return (sigmoid(p[0] * x + p[1]) - y) ** 2

gradcheck(loss_neuron, np.array([w, b]), np.array([dL_dw, dL_db]), name="单神经元")

**PyTorch 对照**：用 `torch.autograd` 对同一表达式自动反向，对比梯度。

In [ ]:
import torch

wt = torch.tensor(w, requires_grad=True)
bt = torch.tensor(b, requires_grad=True)
Lt = (torch.sigmoid(wt * x + bt) - y) ** 2
Lt.backward()

print(f"PyTorch: dL/dw={wt.grad.item():.4f}, dL/db={bt.grad.item():.4f}")
print(f"手推   : dL/dw={dL_dw:.4f}, dL/db={dL_db:.4f}")

## 从零手写实现（二）：两层神经网络

现在升级到一个真正的小网络（向量化，用矩阵运算）：

$$Z_1 = XW_1 + b_1,\quad H = \tanh(Z_1),\quad O = HW_2 + b_2,\quad L = \text{MSE}(O, Y)$$

反向传播时，关键的两个"局部导数"是：

- **矩阵乘法** $C = AB$ 的反向：$\ \partial A = (\partial C)\,B^\top,\quad \partial B = A^\top(\partial C)$
- $\tanh$ 的反向：$\ \tanh'(z) = 1 - \tanh^2(z)$

（matmul 的这条反向规则非常重要，Part 3 造张量引擎时我们会反复用到它。）

In [ ]:
set_seed(0)
N, d_in, d_hid, d_out = 8, 3, 4, 1
X = np.random.randn(N, d_in)
Y = np.random.randn(N, d_out)

W1 = np.random.randn(d_in, d_hid) * 0.5; b1 = np.zeros(d_hid)
W2 = np.random.randn(d_hid, d_out) * 0.5; b2 = np.zeros(d_out)

# ---------- 前向（缓存中间量，反向要用） ----------
Z1 = X @ W1 + b1
H  = np.tanh(Z1)
O  = H @ W2 + b2
L  = np.mean((O - Y) ** 2)
print("loss =", round(float(L), 6))

# ---------- 反向（手推，逐层往回） ----------
dO  = (2.0 / (N * d_out)) * (O - Y)     # L = mean(...) -> 每个元素除以 N*d_out
dW2 = H.T @ dO                          # O = H W2 + b2
db2 = dO.sum(axis=0)
dH  = dO @ W2.T
dZ1 = dH * (1 - np.tanh(Z1) ** 2)       # tanh'
dW1 = X.T @ dZ1                         # Z1 = X W1 + b1
db1 = dZ1.sum(axis=0)
print("梯度形状:", dW1.shape, db1.shape, dW2.shape, db2.shape)

**验证一：数值梯度检查**——逐个参数（`W1, b1, W2, b2`）核对手推梯度。

In [ ]:
def loss_with(name, value):
    P = {"W1": W1, "b1": b1, "W2": W2, "b2": b2}
    P[name] = value
    H_ = np.tanh(X @ P["W1"] + P["b1"])
    O_ = H_ @ P["W2"] + P["b2"]
    return np.mean((O_ - Y) ** 2)

grads = {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}
values = {"W1": W1, "b1": b1, "W2": W2, "b2": b2}
ok = True
for name in ["W1", "b1", "W2", "b2"]:
    passed = gradcheck(lambda p, n=name: loss_with(n, p), values[name], grads[name], name=name)
    ok = ok and passed
print("全部通过 ->", ok)

**验证二：PyTorch 对照**——让 `torch.autograd` 求同一网络的梯度，确认与手推一致。

In [ ]:
Xt = torch.tensor(X); Yt = torch.tensor(Y)
W1t = torch.tensor(W1, requires_grad=True); b1t = torch.tensor(b1, requires_grad=True)
W2t = torch.tensor(W2, requires_grad=True); b2t = torch.tensor(b2, requires_grad=True)

Ot = torch.tanh(Xt @ W1t + b1t) @ W2t + b2t
Lt = ((Ot - Yt) ** 2).mean()
Lt.backward()

for name, g_hand, t in [("W1", dW1, W1t), ("b1", db1, b1t), ("W2", dW2, W2t), ("b2", db2, b2t)]:
    diff = np.abs(t.grad.numpy() - g_hand).max()
    print(f"{name}: 手推 vs PyTorch 最大差异 = {diff:.2e}")

## 📦 沉淀进 minitorch

本节仍是**原理铺垫**——重点是让你亲手体会反向传播。我们没有把这些"一次性手推"的代码沉淀进包，因为它们的问题恰恰在于：**换个模型就得重推一遍**。

这正是下一站要解决的痛点。

## 小练习

1. **换激活函数**：把单神经元里的 `sigmoid` 换成 `tanh`（导数 $1-\tanh^2$）或 `relu`（导数为 0/1），重新手推 `dL/dz` 并用 `gradcheck` 验证。
2. **加一层**：把两层网络扩成三层（再插一个隐藏层 + `tanh`），手推新的反向传播，并用 `gradcheck` 全部验证通过。
3. **思考题**：上面两层网络我们写了 6 行反向代码；如果有 50 层呢？这说明了什么？（提示：这就是 Part 2 要造 `Value` 自动求导引擎的动机。）

## 小结 & 下一站

✅ 我们手推了单神经元和两层网络的反向传播，理解了**链式法则 = 上游梯度 × 局部梯度**，并掌握了最关键的 **matmul 反向公式**。

但你应该也感受到了：**每换一个模型就要重新手推梯度，既繁琐又易错**。

**下一站 → Part 2 `03_value_autograd_engine`**：我们将造一个 `Value` 类——它在你做前向计算的同时**自动记录计算图**，然后一句 `.backward()` 就把所有梯度算出来。这就是 PyTorch/TensorFlow 最核心的魔法，而我们要亲手把它造出来。